# Phase 1 (GitHub + Read + Q&A)

In [22]:
from dotenv import load_dotenv
import os
from pathlib import Path
from agents.mcp import MCPServerStreamableHttp
from agents import Agent, trace, Runner, function_tool
from IPython.display import display, Markdown
import json
import subprocess
import time
import threading
from openai import OpenAI
import traceback
from pydantic import BaseModel, Field

In [23]:
load_dotenv(override=True)

openai = OpenAI()

GITHUB_PAT = os.getenv("GITHUB_PAT")

### Defining MCP Servers

In [24]:
NB_MCP_MEMORY_PATH = Path.cwd() / "memory.jsonl"

github_params = {
    "url": "https://api.githubcopilot.com/mcp/",
    "headers": {"Authorization": f"Bearer {GITHUB_PAT}"},
}

In [25]:
github_mcp_server = MCPServerStreamableHttp(
    params=github_params, client_session_timeout_seconds=30
)

await github_mcp_server.connect()

### Check If Already Cloned Tool

In [26]:
@function_tool
def check_if_already_cloned(repo_url: str) -> str:
    """It checks if repo is already cloned or not.
    Args:
            repo_url: It takes repo_url or repo_name
    """
    clone_dir = "./coding_agent/repos"

    repo_name = repo_url.split("/")[-1].replace(".git", "")
    target_path = Path(clone_dir) / repo_name

    if target_path.exists():
        return f"Repo already exists at {target_path}"
    else:
        return "It is not cloned"

### Cloning Tool

In [27]:
@function_tool
def clone_repo(repo_url: str, branch: str = None):
    """
    Clone a GitHub repository locally.

    :param repo_url: HTTPS or SSH repo URL
    :param clone_dir: Directory where repo will be cloned
    :param branch: Optional branch name
    """

    # Hardcoded path of cloning
    clone_dir = "./coding_agent/repos"

    os.makedirs(clone_dir, exist_ok=True)

    repo_name = repo_url.split("/")[-1].replace(".git", "")
    target_path = os.path.join(clone_dir, repo_name)

    if os.path.exists(target_path):
        print(f"Repo already exists at {target_path}")
        return target_path

    cmd = ["git", "clone"]

    if branch:
        cmd += ["-b", branch]

    cmd += [repo_url, target_path]

    try:
        subprocess.run(cmd, check=True)
        print(f"Cloned into {target_path}")
        return target_path
    except subprocess.CalledProcessError as e:
        print("Error cloning repo:", e)
        return None

### Building Index Tool

##### Summary Builder Function

In [28]:
def summary_builder(index_path: str) -> str:
    """Creates summary of files of repository.
    Args:
            index_path: Path of index.
    Returns:
            Status
    """
    try:
        print("Building summaries in Index . . . ")

        with open(index_path, "r") as repo:
            json_repo = repo.read()
            list_repo = json.loads(json_repo)

        for file in list_repo:
            if file["summary"] is None:
                with open(file["file_path"], "r", encoding="latin-1") as code:
                    snippit = code.read()[:2000]
                    summary = openai.chat.completions.create(
                        model="gpt-4.1-nano",
                        messages=[
                            {
                                "role": "system",
                                "content": "Summarize the purpose of this code file briefly.",
                            },
                            {"role": "user", "content": snippit},
                        ],
                    )

                file["summary"] = summary.choices[0].message.content

        with open(index_path, "w") as f:
            json.dump(list_repo, f, indent=2)

        return "success"

    except Exception as e:
        return f"Failed with an error {str(e)}"

In [29]:
@function_tool
def index_builder(repo_name: str) -> str:
    """Builds index of repositroy in JSON format, which cointains
            - file_name
            - file_path
            - summary
    Args:
            repo_name: The name of repository.
    Returns:
            It creates the index.
    """
    index_path = Path.cwd() / "coding_agent" / f"index_of_{repo_name}.json"

    if not index_path.exists():
        print("Building Index")

        index = []

        repo_path = Path.cwd() / "coding_agent" / "repos" / repo_name

        if repo_path.exists():
            files = list(repo_path.rglob("*"))

            # This removed all empty folders and .git files
            relevent_files = [
                i for i in files if i.suffix != "" and ".git" not in i.parts
            ]

            for file in relevent_files:
                index.append(
                    {"name": file.name, "file_path": str(file), "summary": None}
                )

            with open(
                Path.cwd() / "coding_agent" / f"index_of_{repo_name}.json", "w"
            ) as file:
                json.dump(index, file, indent=2)

            print("Index built")

            summary_status = summary_builder(
                Path.cwd() / "coding_agent" / f"index_of_{repo_name}.json"
            )

            return {"index_status": "success", "summary_status": summary_status}
        else:
            return "This repo doesn't exists"

    else:
        return "Index already exists"

### Index Retriever Tool

In [30]:
class ReleventFile(BaseModel):
    file_name: str = Field("Name of the relevent file")
    file_path: str = Field("File path")
    reason: str = Field("Reason for selecting this file as relevent")


class ReleventFiles(BaseModel):
    relevent_files: list[ReleventFile]

In [ ]:
@function_tool
def file_filter(query: str, repo_name: str) -> str:
    """Used for retrieving the relevent file names with reason.
    Args:
            query: query of user
            repo_name: Exact name of repository
    Returns:
            Relevent file names with path and reason.
    """
    index_path = Path.cwd() / "coding_agent" / f"index_of_{repo_name}.json"
    if index_path.exists():
        with open(index_path, "r") as f:
            content = f.read()
            content = json.loads(content)

        p = []
        for i in content:
            p.append(
                f"Name: {i['name']}\nFile Path: {i['file_path']}\nSummary: {i['summary']}"
            )

        system_prompt = f"""
		You are given a list of files from a codebase.

		Each file has:
		- file_name
 		- file_path
		- summary

		Files:
		{p}

		Select the most relevant files.
		Return their name and file path and reason.
		"""

        summary = openai.chat.completions.parse(
            model="gpt-4.1-nano",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": query},
            ],
            response_format=ReleventFiles,
        )

        return summary.choices[0].message.parsed

    else:
        return "Index does not exists"

### Code Reading Tool

In [32]:
@function_tool
def read_code(files: list[str]) -> str:
    """Returns the whole code of the file.
    Args:
            files: list[str]: It takes the list of the names of the relevent files
    Returns:
            Gives the whole code for each file
    """
    code_formatted = ""

    for file in files:
        if len(code_formatted) > 15000:
            code_formatted += "\n[Truncated due to size]\n"
            break
        path = next(Path(Path.cwd() / "coding_agent").rglob(file))
        if path.exists():
            try:
                with open(path, "r", encoding="utf-8") as f:
                    code = f.read()
                    code_formatted += f"{path}:\n{code}\n{'-' * 30}\n"
            except Exception as e:
                code_formatted += (
                    f"{path}: [Error reading file: {str(e)}]\n{'-' * 30}\n"
                )
        else:
            code_formatted += f"{path}: [File not found]\n{'-' * 30}\n"
    return code_formatted

NameError: name 'function_tool' is not defined

### Defining Agents

In [ ]:
github_instructions = "You are smart coding agent which helps users to work on code."

cloning_instructions = """If user wants to do some analysis or Q&A and wants you to work on the code, bugs or some feature request. 
Then you should first always run the tool 'check_if_already_cloned' to check if repo is there or not. If it not there, then you should always tell user that 
first you have to clone the repo, and ask his permission for cloning. And if you have already cloned it, then you don't have to call this tool. 
But before asking anything from user, you should always first run the tool 'check_if_already_cloned'.

Below are the instructions for cloning the repo,
Always call 'clone_repo' tool. You can confirm this from user if he wants to clone it. 
In this you have to pass 2 arguments, out of which one is optional
1. URL of the repo: You need to pass full URL of the repo (This can be made "https://github.com/{{username}}/{{repo_name}}))
2. Branch (Optional): If user specifies the branch name then paste there, otherwise you can leave it. Never asks the branch name explicitly.
"""

next_steps_instructions = """
Once cloned, you immedietly have to make an index of the repo by calling 'index_builder' tool, this is a MUST step. 
This tool takes only one argument, and that is repo name as it is. Be very careful on this. 
It will create a JSON file. And it'll store, file name, file path, and the description of each file of the repo.
And remember, if you have already built index of this same repo, then do not call this tool again and again.

Now whenver user asks you to work on any file or bug or asks you anything, you simply have to look in this index to determine which file you need to read, from there you can fetch the path.
"""

retriever_instructions = """
For searching files, and getting their quick description and path, always call this tool 'file_filter' for retrieving the file name with description. 
This tool will take 2 arguments,
1. query: It could be user's query or what kind of file are you searching
2. repo_name: Name of the repositrory as it is (be very careful on this)
"""

code_reading_instructions = """
Now whenever users asks for any Q&A, like summerising, analyzing some file or code, you need to call 'file_filter' tool, as it will give you the file's path. You must retrieve names of these files.
Then you must have to call 'read_code' tool, as this will return the code of that file(s). In this you need to pass the list of names of files, these names will come from 'file_filter' tool. 
And then this tool will give you the code. 
"""

instructions = f"""{github_instructions}

{cloning_instructions}

{retriever_instructions}

{code_reading_instructions}
"""

In [35]:
github_agent = Agent(
    name="Github agent",
    instructions=instructions,
    model="gpt-4.1-mini",
    tools=[check_if_already_cloned, clone_repo, index_builder, file_filter, read_code],
    mcp_servers=[github_mcp_server],
)

# with trace("Github test"):
# 	result = await Runner.run(github_agent, "Can you clone my recruiters_email_reply_agent repo")

# display(Markdown(result.final_output))

### Implementing Memory

In [36]:
MEMORY_FILE = str(Path.cwd() / "chat_memory.json")


def load_memory():
    if not os.path.exists(MEMORY_FILE):
        return {
            "chat_history": [],
            # "repo_context": {}
        }
    with open(MEMORY_FILE, "r") as f:
        return json.load(f)


def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

In [37]:
def build_context(memory, user_input):
    chat_history = memory["chat_history"][-5:]

    context = ""

    for msg in chat_history:
        context += f"{msg['role']}: {msg['content']}\n"

    # Add repo ONLY when needed
    #  if intent in ["repo_query", "repo_task"]:
    #   context += f"\nRepo Context:\n{memory['repo_context']}\n"

    context += f"\nUser: {user_input}"

    return context

## Running our Agent

In [38]:
def spinner(stop_event, display_handle):
    i = 0
    animation = ["o.o0o.o", ".o.O.o.", "oO.o.Oo", "Oo...oO"]
    while not stop_event.is_set():
        display_handle.update(
            Markdown(f"**Agent:** {animation[i % len(animation)] * 2}")
        )
        time.sleep(0.2)
        i += 1

In [39]:
async def main_agent_loop():
    memory = load_memory()
    c = 0

    while c < 20:
        user_input = input("User: ")
        display(Markdown(f"**User:** {user_input}"))

        context = build_context(memory, user_input)

        result = None
        stop_loading = None
        thread = None
        spinner_display_handle = None

        try:
            stop_loading = threading.Event()
            spinner_display_handle = display(
                Markdown("Starting Agent"), display_id=True
            )

            thread = threading.Thread(
                target=spinner, args=(stop_loading, spinner_display_handle)
            )
            thread.start()

            with trace("Coding agent"):
                result = await Runner.run(github_agent, context)

        except Exception:
            traceback.print_exc()

        finally:
            if stop_loading:
                stop_loading.set()
            if thread:
                thread.join()

        if result is not None:
            if spinner_display_handle:
                spinner_display_handle.update(
                    Markdown(f"**Agent:** {result.final_output}")
                )
            else:
                print("Agent:", result.final_output)

            memory["chat_history"].append({"role": "user", "content": user_input})
            memory["chat_history"].append(
                {"role": "assistant", "content": result.final_output}
            )
        else:
            if spinner_display_handle:
                spinner_display_handle.update(Markdown("**Agent:** Error occurred"))
            else:
                print("Agent: Error occurred")

        memory["chat_history"] = memory["chat_history"][-10:]
        save_memory(memory)

        c += 1

In [ ]:
await main_agent_loop()

[non-fatal] Tracing client error 400: {
  "error": {
    "message": "Invalid type for 'data[1].span_data.result': expected an array of strings, but got null instead.",
    "type": "invalid_request_error",
    "param": "data[1].span_data.result",
    "code": "invalid_type"
  }
}


**User:**     What does this repo do?

**Agent:** The repository "recruiters_email_reply_agent" appears to be an email automation system focused on handling recruitment-related emails. From the code and project files, here is what the repo does:

- It fetches the latest emails from Gmail (agent_1).
- It classifies emails to identify relevant job opportunities (agent_2).
- It retrieves the full email content when needed for detailed processing (agent_3).
- It generates draft email responses automatically based on the analysis (agent_4).
- It writes/saves those draft replies back into Gmail as email drafts (agent_5).
- The core pipeline orchestrates these steps in sequence, providing an end-to-end flow for automated email reply handling.

Main features supported:
- Email fetching from Gmail.
- Email classification for relevance (likely for recruitment/job opportunities).
- Draft reply generation using AI or other logic.
- Draft saving in Gmail for user review and sending.

The repository is configured as a Python project with dependencies like `google-api-python-client` for Gmail access, `openai` for AI-driven processing, and other utilities.

If you would like, I can provide a deeper overview of specific parts, e.g., how the pipeline works or how drafts are generated and written. Would you like me to do that?

**User:**     “Did we already clone it?”

**Agent:** Yes, the repository "recruiters_email_reply_agent" is already cloned and available for access. How would you like to proceed? Do you want me to analyze specific files or implement any changes?

...

**Agent:** API calls in the "recruiters_email_reply_agent" repository are made in several key files related to email and OpenAI services:

1. pyproject.toml - Specifies dependencies for making API calls, including OpenAI libraries.
2. settings.py (src/config/settings.py) - Retrieves the API key for OpenAI.
3. gmail_client.py (src/tools/gmail/gmail_client.py) - Manages interactions with Gmail API for email data access.
4. fetch_latest_emails.py (src/tools/fetch_latest_emails.py) - Performs API calls to Gmail to fetch the latest emails.
5. fetch_full_emails.py (src/tools/fetch_full_emails.py) - Retrieves full email details from Gmail via API.
6. write_drafts_in_gmail.py (src/tools/write_drafts_in_gmail.py) - Creates email draft responses through Gmail API calls.

Do you want me to open and show the code details of API calls in any specific file?

**User:**     Open file settings.py

**Agent:** The file src/config/settings.py contains the following code:

```python
import os

openai_key = os.getenv("OPENAI_API_KEY")
```

This file fetches the OpenAI API key from the environment variable "OPENAI_API_KEY". It is used for authenticating API calls to OpenAI services.

Would you like me to open any other files where API calls are made?

**User:** can you list down tools you have

**Agent:** The tools I have for working with the "recruiters_email_reply_agent" repository include:

1. pyproject.toml - Dependency specification for API interaction.
2. src/config/settings.py - Configuration to retrieve API keys.
3. src/tools/gmail/gmail_client.py - Handles Gmail API for email management.
4. src/tools/fetch_latest_emails.py - Fetches the latest emails using Gmail API.
5. src/tools/fetch_full_emails.py - Retrieves full email details through Gmail API.
6. src/tools/write_drafts_in_gmail.py - Creates draft replies in Gmail using API.

These tools involve API calls to interact with Gmail and OpenAI services.

Would you like me to open and show the code for any specific tool or file? Or do you want me to find specific API calls in them?